### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [6]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

### Data Ingestion And Processing


In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

text splitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(sample_documents)
print(chunks)


[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognit

In [7]:
embedding=OllamaEmbeddings(model="llama3.2:latest")
embedding.embed_query("What is machine learning?")

[0.014320511,
 0.010946855,
 -0.002557826,
 -0.016597148,
 0.0064507625,
 -0.012299443,
 0.021673765,
 -0.008057453,
 0.008713857,
 -0.0070217177,
 0.0020326022,
 0.0057243896,
 0.0014786732,
 0.01704144,
 -0.016937863,
 0.01718521,
 -0.012074988,
 0.0020705468,
 0.00052593637,
 -0.015173176,
 0.0040690405,
 0.012221786,
 0.012971886,
 -0.0006800548,
 0.004179693,
 -0.016282104,
 0.013269296,
 -0.038469847,
 0.013761651,
 -0.011560305,
 -0.018940482,
 -0.0023283036,
 0.025537401,
 0.016355723,
 0.0043751653,
 -0.015389573,
 -0.0038937414,
 0.008385541,
 -0.0044490187,
 -0.017628808,
 -0.011421576,
 0.014392887,
 0.01298601,
 0.009653226,
 -0.003383289,
 0.010218991,
 0.017560305,
 -0.002435824,
 0.00010778045,
 -0.009142663,
 0.020465674,
 -0.004532482,
 -0.0064843544,
 0.024389174,
 0.0024590036,
 0.0018282598,
 0.019713687,
 -0.029942125,
 0.023911899,
 -0.00017456134,
 -0.023393784,
 0.00041014559,
 -0.0013823811,
 -0.022522712,
 0.002748886,
 -0.02053516,
 0.023600943,
 -0.01394698

In [8]:
def compare_embeddings(text1,text2):
    emb1=embedding.embed_query(text1)
    emb2=embedding.embed_query(text2)
    cosine_sim=np.dot(emb1,emb2)/(np.linalg.norm(emb1)*np.linalg.norm(emb2))
    return cosine_sim
similarity=compare_embeddings("What is machine learning?","Explain machine learning")
print(f"Cosine Similarity: {similarity}")


Cosine Similarity: 0.5721044721422238


## Faiss Vectorstore

In [11]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)
print(f"Number of documents in vector store: {vectorstore.index.ntotal}")

Number of documents in vector store: 4


In [12]:
vectorstore.save_local("faiss_index")
print("Vector store saved locally as 'faiss_index' directory.")

Vector store saved locally as 'faiss_index' directory.


load

In [15]:
loaded_vectorstore=FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)
print(f"Number of documents in loaded vector store: {loaded_vectorstore.index.ntotal}")

Number of documents in loaded vector store: 4


similarity Search

In [16]:
query='what is deep learning'
results=vectorstore.similarity_search(query,k=2)
print(f"Top 2 results for query '{query}':")

Top 2 results for query 'what is deep learning':


In [17]:
print("querry:",query)
for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

querry: what is deep learning
Result 1:
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
--------------------------------------------------
Result 2:
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
Metadata: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
--------------------------------------------------


similarity search with score

In [19]:
results_similarity_search_with_score=vectorstore.similarity_search_with_score(query,k=2)
print(f"Top 2 results with scores for query '{query}':")
for i, (doc, score) in enumerate(results_similarity_search_with_score, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print(f"Similarity Score: {score:.4f}")
    print("-" * 50)

Top 2 results with scores for query 'what is deep learning':
Result 1:
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
Similarity Score: 1.2207
--------------------------------------------------
Result 2:
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
Metadata: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
Similarity Score: 1.3563
--------------------------------------------------


metadata filtering

In [20]:
filter_data={"topic":"ML"}
filtered_results=vectorstore.similarity_search(query,k=2,filter=filter_data)
print(f"Top 2 filtered results for query '{query}' with filter {filter_data}:")
for i, doc in enumerate(filtered_results, 1):
    print(f"Result {i}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

Top 2 filtered results for query 'what is deep learning' with filter {'topic': 'ML'}:
Result 1:
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
Metadata: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
--------------------------------------------------
